# Train a YOLO26n baseline

Fine-tune the small pretrained Ultralytics YOLO26 model on this workspace's YOLO-format dataset. The dataset YAML uses paths relative to the workspace root. Training and validation use the `train` and `valid` splits; the `test` split is reserved for final evaluation.

In [3]:
%pip install ultralytics

  Using cached ultralytics-8.4.211-py3-none-any.whl.metadata (47 kB)
  Using cached matplotlib-3.11.2-cp313-cp313-win_amd64.whl.metadata (80 kB)
  Using cached torchvision-0.29.2-py3-none-win_amd64.whl.metadata (5.7 kB)
  Using cached polars-2.1.0-py3-none-any.whl.metadata (11 kB)
  Using cached ultralytics_thop-2.2.5-py3-none-any.whl.metadata (14 kB)
  Using cached ultralytics_platform-0.1.110-py3-none-any.whl.metadata (18 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached anyio-4.16.0-py3-none-any.whl.metadata (4.7 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
Using cached ultralytics-8.4.211-py3-none-any.whl (1.5 MB)
Using cached matplotlib-3.11.2-cp313-cp313-win_amd64.whl (9.3 MB)
Using cached polars-2.1.0-py3-none-any.whl (911 kB)
Using cached torchvision-0.29.2-py3-none-win_amd64.whl (1.4 MB)
Using cached ultralytics_platform-0.1.110-py3-none-any.whl (103 kB)
Using cached ultralytics_thop-2.2.5-py3-none-any.whl (32 kB)
Using ca

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import torch
import yaml

ROOT = Path.cwd()
if not (ROOT / "data.yaml").is_file() and (ROOT.parent / "data.yaml").is_file():
    ROOT = ROOT.parent

DATA_YAML = ROOT / "data.yaml"
if not DATA_YAML.is_file():
    raise FileNotFoundError(f"Could not find data.yaml from {Path.cwd()}")

with DATA_YAML.open(encoding="utf-8") as f:
    data_config = yaml.safe_load(f)

for split in ("train", "val", "test"):
    image_dir = Path(data_config[split])
    if not image_dir.is_absolute():
        image_dir = DATA_YAML.parent / image_dir
    if not image_dir.is_dir():
        raise FileNotFoundError(f"Missing {split} images: {image_dir}")
    print(f"{split}: {len(list(image_dir.glob('*')))} files in {image_dir}")

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print(f"PyTorch: {torch.__version__}; CUDA available: {torch.cuda.is_available()}; device: {DEVICE}")

train: 2099 files in d:\AI-TOD.v2i.yolo26\train\images
val: 600 files in d:\AI-TOD.v2i.yolo26\valid\images
test: 300 files in d:\AI-TOD.v2i.yolo26\test\images
PyTorch: 2.15.0+cpu; CUDA available: False; device: cpu


## Train

This baseline uses pretrained `yolo26n.pt`, 50 epochs, and 640-pixel images. The training cell also initializes its dataset and device settings so it can be run after a kernel restart. If training is interrupted, run the cell again to resume from the latest saved `last.pt` checkpoint (the last completed epoch). On a CUDA machine it selects the GPU; otherwise it runs on CPU and may take considerably longer. The best checkpoint and training metrics are saved under `runs/detect/yolo26n_baseline`.

In [5]:
from pathlib import Path
import torch
import yaml
from ultralytics import YOLO

ROOT = Path.cwd()
if not (ROOT / "data.yaml").is_file() and (ROOT.parent / "data.yaml").is_file():
    ROOT = ROOT.parent
DATA_YAML = ROOT / "data.yaml"
if not DATA_YAML.is_file():
    raise FileNotFoundError(f"Could not find data.yaml from {Path.cwd()}")
DEVICE = 0 if torch.cuda.is_available() else "cpu"
RUN_DIR = ROOT / "runs" / "detect" / "yolo26n_baseline"
LAST_CHECKPOINT = RUN_DIR / "weights" / "last.pt"

if LAST_CHECKPOINT.is_file():
    print(f"Resuming from {LAST_CHECKPOINT}")
    model = YOLO(str(LAST_CHECKPOINT))
    results = model.train(resume=True)
else:
    print("No resume checkpoint found; starting from pretrained yolo26n.pt")
    model = YOLO("yolo26n.pt")
    results = model.train(
        data=str(DATA_YAML),
        epochs=100,
        imgsz=640,
        batch=8,
        device=DEVICE,
        workers=0,
        project=str(ROOT / "runs" / "detect"),
        name="yolo26n_baseline",
        exist_ok=True,
        patience=10,
        seed=42,
        plots=True,
    )
print(f"Training outputs: {results.save_dir}")

Resuming from d:\AI-TOD.v2i.yolo26\runs\detect\yolo26n_baseline\weights\last.pt
New https://pypi.org/project/ultralytics/8.4.212 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.211  Python-3.13.3 torch-2.15.0+cpu CPU (13th Gen Intel Core i5-1335U)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=d:\AI-TOD.v2i.yolo26\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.00111

KeyboardInterrupt: 

## Final test-set evaluation

Run this cell after training completes. It evaluates the best validation checkpoint on the held-out `test` split.

In [ ]:
best_weights = Path(results.save_dir) / "weights" / "best.pt"
if not best_weights.is_file():
    raise FileNotFoundError(f"Best checkpoint was not created: {best_weights}")

best_model = YOLO(str(best_weights))
test_metrics = best_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    batch=8,
    device=DEVICE,
    workers=0,
)
print(test_metrics.results_dict)